# Experiment 6: Model Registry and Version Management

## AIM:
Implement model registration and lifecycle management by maintaining model versions across staging, validation, and production stages. Track model metadata, preprocessing dependencies, evaluation metrics, dataset lineage, and deployment readiness, and perform model comparison, promotion, rollback, and traceability throughout the ML lifecycle.

---
### Objectives:
1. **Model Registration**: Register Scikit-Learn pipelines into the MLflow Model Registry (`WalmartSalesForecastingModel`).
2. **Lifecycle Stages & Aliases**: Maintain model versions across `Staging`, `Production`, `Archived`, and aliases (`champion`, `challenger`).
3. **Automated Promotion**: Compare a candidate challenger against the active production champion on holdout test data; automate promotion if threshold criteria are met.
4. **Deployment Smoke Test**: Execute deployment readiness checks (load latency, inference stability, absence of NaNs).
5. **Rollback Mechanism**: Demonstrate safe automated rollback to restore the previous stable production model.
6. **End-to-End Lineage**: Trace full provenance from Dataset -> Run ID -> Model Version -> Production Stage.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

if ".." not in sys.path:
    sys.path.insert(0, "..")

from src.pipeline import build_full_model_pipeline
from src.registry_manager import ModelRegistryManager

print("Libraries imported successfully. MLflow version:", mlflow.__version__)


## Step 1: Initialize Model Registry Manager
We connect to our local SQLite backend (`sqlite:///../mlflow.db`) and target the registered model entity `WalmartSalesForecastingModel`.


In [ ]:
tracking_uri = "sqlite:///../mlflow.db"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("Experiment_6_Model_Registry_Lifecycle")

model_name = "WalmartSalesForecastingModel"
registry = ModelRegistryManager(model_name=model_name, tracking_uri=tracking_uri)
print(f"Connected to Model Registry for entity: '{model_name}'")


## Step 2: Prepare Holdout Evaluation Data


In [ ]:
data_path = os.path.join("..", "Walmart.csv")
raw_df = pd.read_csv(data_path)
X = raw_df.drop(columns=["Weekly_Sales"])
y = raw_df["Weekly_Sales"]

SEED = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED)
input_example = X_train.head(3)
signature = infer_signature(input_example, y_train.head(3))

print(f"Evaluation holdout set: {len(X_test)} samples")


## Step 3: Train & Register Model Version 1 (Baseline)
We train an initial baseline Ridge Regression model and register it as Version 1, setting its initial stage to `Production` (alias: `champion`).


In [ ]:
pipeline_v1 = build_full_model_pipeline(
    regressor=Ridge(alpha=10.0, random_state=SEED),
    scaler_type="standard"
)

with mlflow.start_run(run_name="Train_Model_Version_1_Ridge") as run_v1:
    run_v1_id = run_v1.info.run_id
    pipeline_v1.fit(X_train, y_train)
    
    y_pred_v1 = pipeline_v1.predict(X_test)
    rmse_v1 = float(np.sqrt(np.mean((y_test - y_pred_v1) ** 2)))
    r2_v1 = float(pipeline_v1.score(X_test, y_test))
    
    mlflow.log_param("model_family", "Ridge")
    mlflow.log_param("alpha", 10.0)
    mlflow.log_metric("test_rmse", rmse_v1)
    mlflow.log_metric("test_r2", r2_v1)
    
    mlflow.sklearn.log_model(
        sk_model=pipeline_v1,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
        serialization_format="cloudpickle"
    )

# Register as Version in MLflow Registry
mv_1 = registry.register_model_from_run(
    run_id=run_v1_id,
    tags={"architecture": "Linear_Ridge", "training_stage": "initial_baseline"}
)

# Transition to Production
registry.transition_stage(mv_1.version, "Production", archive_existing=True)
registry.set_alias(mv_1.version, "champion")

print(f"Registered Model Version {mv_1.version} promoted to 'Production' (alias='champion')")
print(f"Baseline Test RMSE: ${rmse_v1:,.2f} | R²: {r2_v1:.4f}")


## Step 4: Train & Register Model Version 2 (Challenger)
Next, an automated CI/CD pipeline trains a higher-capacity Random Forest Regressor and registers it as Version 2, placing it in `Staging` (alias: `challenger`).


In [ ]:
pipeline_v2 = build_full_model_pipeline(
    regressor=RandomForestRegressor(n_estimators=100, max_depth=14, random_state=SEED, n_jobs=-1),
    scaler_type="standard"
)

with mlflow.start_run(run_name="Train_Model_Version_2_RandomForest") as run_v2:
    run_v2_id = run_v2.info.run_id
    pipeline_v2.fit(X_train, y_train)
    
    y_pred_v2 = pipeline_v2.predict(X_test)
    rmse_v2 = float(np.sqrt(np.mean((y_test - y_pred_v2) ** 2)))
    r2_v2 = float(pipeline_v2.score(X_test, y_test))
    
    mlflow.log_param("model_family", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 14)
    mlflow.log_metric("test_rmse", rmse_v2)
    mlflow.log_metric("test_r2", r2_v2)
    
    mlflow.sklearn.log_model(
        sk_model=pipeline_v2,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
        serialization_format="cloudpickle"
    )

# Register Version 2
mv_2 = registry.register_model_from_run(
    run_id=run_v2_id,
    tags={"architecture": "Random_Forest_Ensemble", "training_stage": "tuned_challenger"}
)

# Place in Staging
registry.transition_stage(mv_2.version, "Staging", archive_existing=False)
registry.set_alias(mv_2.version, "challenger")

print(f"Registered Model Version {mv_2.version} placed in 'Staging' (alias='challenger')")
print(f"Challenger Test RMSE: ${rmse_v2:,.2f} | R²: {r2_v2:.4f}")


## Step 5: Automated Model Comparison & Promotion
We execute `compare_and_promote()`. The function loads both models, computes holdout RMSE, and promotes the candidate to `Production` only if it beats the current production champion by at least $5.0\%$.


In [ ]:
promotion_decision = registry.compare_and_promote(
    candidate_version=mv_2.version,
    X_val=X_test,
    y_val=y_test,
    improvement_threshold_pct=5.0
)

print("=== PROMOTION DECISION REPORT ===")
print(f"Candidate Version          : {promotion_decision['candidate_version']}")
print(f"Candidate Test RMSE        : ${promotion_decision['candidate_metrics']['rmse']:,.2f}")
print(f"Current Production Version : {promotion_decision.get('current_prod_version')}")
print(f"Current Prod Test RMSE     : ${promotion_decision['current_prod_metrics']['rmse']:,.2f}")
print(f"Observed RMSE Improvement  : {promotion_decision['rmse_improvement_pct']}%")
print(f"Promotion Result           : {'PROMOTED' if promotion_decision['promoted'] else 'REJECTED'}")
print(f"Audit Rationale            : {promotion_decision['reason']}")


## Step 6: Deployment Readiness & Smoke Testing
Before opening live traffic, we run automated deployment readiness checks on the active production model:
- Load latency benchmark
- Inference speed test on sample batch
- Absence of NaNs or Infs
- Output range verification


In [ ]:
active_prod = registry.get_version_by_alias_or_stage("champion")
readiness_report = registry.verify_deployment_readiness(
    version=active_prod.version,
    sample_input=X_test.head(10)
)

print("=== DEPLOYMENT READINESS AUDIT ===")
for k, v in readiness_report.items():
    print(f"{k:<30}: {v}")

assert readiness_report["is_ready_for_deployment"], "Production smoke test failed!"
print("
>>> Model is certified READY for serving.")


## Step 7: Automated Rollback Simulation
If live anomalies or regressions occur post-deployment, `rollback_production()` demotes the faulty version and restores the previous champion within milliseconds.


In [ ]:
print("Simulating operational rollback to previous stable version...")
rollback_result = registry.rollback_production()

print(f"Rollback Executed          : {rollback_result['success']}")
print(f"Demoted Faulty Version     : {rollback_result.get('demoted_version')}")
print(f"Restored Production Version: {rollback_result['restored_production_version']}")
print(f"Message                    : {rollback_result['message']}")

# Re-promote Version 2 back to champion so our final serving model is optimal
registry.transition_stage(mv_2.version, "Production", archive_existing=True)
registry.set_alias(mv_2.version, "champion")
print(f"
Final State: Restored Version {mv_2.version} as active Production Champion.")


## Step 8: End-to-End Model Traceability & Lineage
Full governance and compliance require linking every deployed production model back to its training run, parameters, metrics, dataset hash, and author.


In [ ]:
final_prod = registry.get_version_by_alias_or_stage("champion")
lineage = registry.get_model_lineage(final_prod.version)

print(f"Model Name           : {lineage['model_name']}")
print(f"Active Production Ver: {lineage['version']}")
print(f"Source Run ID        : {lineage['run_id']}")
print(f"Lifecycle Tags       : {lineage['model_tags']}")
print(f"Hyperparameters      : {lineage['run_parameters']}")
print(f"Validation Metrics   : {lineage['run_metrics']}")
print("
>>> End-to-End Traceability successfully verified.")
